In [0]:
from pyspark.sql.functions import col 

def comentarTabelas(
    str_endereco_tabela: str, 
    str_des_tabela: str, 
    lst_colunas_comentarios: dict 
): 
        """ Define comentários de tabela e de colunas no Databricks/Spark. 
        Parâmetros: 
        - str_endereco_tabela: 'catalogo.schema.tabela' 
        - str_des_tabela: descrição da tabela (string) 
        - lst_colunas_comentarios: dict {'coluna1': 'comentário1', ...} """ 
        
        def _escape_sql_literal(s: str) -> str: 
            # Escapa aspas simples para uso em literais SQL 
            return s.replace("'", "''") if isinstance(s, str) else s 
        
        # ------------------------------ 
        # Comentário da TABELA 
        # ------------------------------ 
        
        sdf_detail = spark.sql(f"DESCRIBE DETAIL {str_endereco_tabela}") 
        des_atual = (sdf_detail.select("description").first().description) or "" 
        des_nova = (str_des_tabela or "") 
        
        if des_atual != des_nova: 
            des_sql = _escape_sql_literal(des_nova) 
            spark.sql(f"COMMENT ON TABLE {str_endereco_tabela} IS '{des_sql}'") 
            print(f"[Tabela] Comentário atualizado: {str_endereco_tabela}") 
        else: print(f"[Tabela] Comentário mantido: {str_endereco_tabela}") 
        
        # ------------------------------ 
        # Comentários das COLUNAS 
        # ------------------------------ 
        # DESCRIBE retorna linhas de colunas + seções de metadados; manter só colunas reais 
        
        sdf_desc = (spark.sql(f"DESCRIBE {str_endereco_tabela}") 
                    .select("col_name", "data_type", "comment")) 
        
        sdf_colunas = sdf_desc.where(
            (col("data_type").isNotNull()) & (~col("col_name").startswith("#")) 
        ) 
        
        # Mapa col -> comentário atual 
        linhas = sdf_colunas.select("col_name", "comment").collect() 
        comentarios_atuais = {r.col_name: (r.comment or "") for r in linhas} 
        colunas_existentes = list(comentarios_atuais.keys()) 
        
        # Itera apenas sobre colunas existentes na tabela
        for col_name in colunas_existentes: 
            comentario_novo = lst_colunas_comentarios.get(col_name, None) 
            
            # Se não foi fornecido comentário para essa coluna, não faz nada 
            if comentario_novo is None: 
                # Opcional: logue se quiser monitorar colunas sem input 
                #print(f"[Coluna] Sem novo comentário para: {col_name} (mantido)") 
                continue
            
            comentario_atual = comentarios_atuais.get(col_name, "") 
            comentario_novo_norm = comentario_novo or "" 
            
            if comentario_atual != comentario_novo_norm: 
                comentario_sql = _escape_sql_literal(comentario_novo_norm) 
                spark.sql(
                    f"ALTER TABLE {str_endereco_tabela} " 
                    f"ALTER COLUMN {col_name} COMMENT '{comentario_sql}'"
                ) 
                print(f"[Coluna] Atualizado: {col_name}") 
            else: 
                print(f"[Coluna] Mantido: {col_name}")


In [0]:
import re
import unicodedata
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def normalize_text(value):
    if value is None:
        return None

    # Converte para string
    value = str(value)

    # Remove espaços no início e fim
    value = value.strip()

    # Substitui separadores por ":"
    value = value.replace(" -", ":")
    # value = value.replace(":", "/")

    # Padroniza espaços ao redor da barra
    value = re.sub(r"\s*/\s*", " / ", value)

    # Remove múltiplos espaços
    value = re.sub(r"\s+", " ", value)

    # # Remove acentos
    # value = unicodedata.normalize("NFKD", value).encode("ASCII", "ignore").decode("utf-8")

    # Caixa alta
    value = value.upper()

    return value

normalize_text_udf = F.udf(normalize_text, StringType())